In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

threshold = 0.5

In [2]:
# Rebuild the column lists
attack_df = pd.read_csv("../data/processed_data_v2.csv")

method_cols = [c for c in attack_df.columns if c.startswith("method_")]
numeric_cols = ['prompt_words', 'avg_word_len', 'non_ascii_share', 'char_entropy',
                'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4',
                'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9']
capability_cols = ['arena_expert', 'EQ_bench']
bench_cols = capability_cols + ['sorry_bench']

# get_dummies removed generating_method, so rebuild it from the method_ columns
attack_df["generating_method"] = attack_df[method_cols].idxmax(axis=1).str.removeprefix("method_")
attack_df["attack_and_llm"] = attack_df["generating_method"] + " / " + attack_df["model"]

# same train test split as cv_v2
train_df, test_df = train_test_split(attack_df, test_size=0.2, random_state=120)
y_train = train_df["jailbroken"].astype(int)
y_test = test_df["jailbroken"].astype(int)

scale_cols = numeric_cols + bench_cols
scaler = StandardScaler()
train_scaled = train_df.copy()
test_scaled = test_df.copy()
train_scaled[scale_cols] = scaler.fit_transform(train_df[scale_cols])
test_scaled[scale_cols] = scaler.transform(test_df[scale_cols])
train_scaled[scale_cols] = np.clip(train_scaled[scale_cols], -2, 2)
test_scaled[scale_cols] = np.clip(test_scaled[scale_cols], -2, 2)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

splits = {
    "train":      (train_scaled, y_train),
    "test":       (test_scaled, y_test),
}


# %% Models and metrics
# Update these settings if the grid search in cv_v2 picks different ones after the clipping fix
models = {
    "Decision tree":     DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=120),
    "Random forest":     RandomForestClassifier(n_estimators=300, max_features=0.1, min_samples_leaf=5,
                                                random_state=120, n_jobs=-1),
    "Gradient boosting": GradientBoostingClassifier(learning_rate=0.05, max_depth=4, n_estimators=100,
                                                    random_state=120),
    "XGBoost":           XGBClassifier(learning_rate=0.05, max_depth=3, n_estimators=300, random_state=120),
}


def get_metrics(y_true, y_score):
    """Same metrics as fold_metrics in cv_v2."""
    y_pred = (y_score >= threshold).astype(int)
    return {"accuracy":  accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall":    recall_score(y_true, y_pred),
            "f1":        f1_score(y_true, y_pred, zero_division=0),
            "auc":       roc_auc_score(y_true, y_score)}


# %% Item 1: how do the models do with fewer features?
feature_sets = {
    "All features":             numeric_cols + bench_cols + method_cols,
    "No sorry_bench":           numeric_cols + capability_cols + method_cols,
    "No benchmarks":            numeric_cols + method_cols,
    "No method columns":        numeric_cols + bench_cols,
}

rows = []
for set_name, features in feature_sets.items():
    for model_name, model in models.items():
        model.fit(train_scaled[features], y_train)
        for split_name, (X, y) in splits.items():
            y_score = model.predict_proba(X[features])[:, 1]
            rows.append({"features": set_name, "model": model_name, "split": split_name,
                         **get_metrics(y, y_score)})
results = pd.DataFrame(rows)


def show(split_name, metric):
    """One line per feature set, one column per model."""
    table = results[results["split"] == split_name].pivot(index="features", columns="model", values=metric)
    print(f"\n{split_name}: {metric}")
    print(table.loc[list(feature_sets), list(models)].round(3).to_string())


for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    show("test", metric)

# %% Item 1: simple baselines
# Score each test row with its group's jailbreak rate on the training rows
baselines = {"Attack method": "generating_method",
             "Attack + LLM":  "attack_and_llm"}
 
baseline_rows = []
for name, group_col in baselines.items():
    rate = y_train.groupby(train_df[group_col]).mean()
    y_score = test_df[group_col].map(rate)
    baseline_rows.append({"baseline": name, **get_metrics(y_test, y_score)})
 
print("\nBaselines on the test rows")
print(pd.DataFrame(baseline_rows).round(3).to_string(index=False))

Train rows: 1309
Test rows: 328

test: accuracy
model              Decision tree  Random forest  Gradient boosting  XGBoost
features                                                                   
All features               0.866          0.866              0.890    0.866
No sorry_bench             0.866          0.866              0.878    0.878
No benchmarks              0.793          0.787              0.793    0.793
No method columns          0.805          0.866              0.875    0.866

test: precision
model              Decision tree  Random forest  Gradient boosting  XGBoost
features                                                                   
All features               0.844          0.854              0.877    0.840
No sorry_bench             0.844          0.850              0.871    0.860
No benchmarks              0.846          0.825              0.834    0.838
No method columns          0.861          0.850              0.866    0.847

test: recall
model    

In [5]:
# %% Item 2: gradient boosting results for each attack method and LLM
# Change this line to look at a different feature set
best_features = feature_sets["All features"]
 
best_model = models["Gradient boosting"]
best_model.fit(train_scaled[best_features], y_train)
 
 
def metrics_by_method_and_llm(X, y):
    """One row per attack method and LLM pair."""
    predictions = X[["generating_method", "model"]].copy()
    predictions["actual"] = y
    predictions["predicted"] = (best_model.predict_proba(X[best_features])[:, 1] >= threshold).astype(int)
 
    rows = []
    for (method, llm), group in predictions.groupby(["generating_method", "model"]):
        actual, predicted = group["actual"], group["predicted"]
        rows.append({"generating_method": method, "model": llm,
                     "f1":        f1_score(actual, predicted, zero_division=np.nan),
                     "accuracy":  accuracy_score(actual, predicted),
                     "precision": precision_score(actual, predicted, zero_division=np.nan),
                     "recall":    recall_score(actual, predicted, zero_division=np.nan)})
    return pd.DataFrame(rows)
 
 
# NaN means the score can't be calculated:
#   every metric: no rows for this attack and LLM
#   recall:       no real jailbreaks
#   precision:    nothing was flagged as a jailbreak
#   f1:           no real jailbreaks and nothing was flagged
llm_order = ["llama", "vicuna", "gpt3.5", "gpt4"]
 
for split_name, (X, y) in splits.items():
    if split_name == "train":
        continue
    by_group = metrics_by_method_and_llm(X, y)
    for metric in ["f1", "accuracy", "precision", "recall"]:
        table = by_group.pivot(index="generating_method", columns="model", values=metric)
        print(f"\nGradient boosting, {split_name}: {metric}")
        print(table[llm_order].round(3).to_string())


Gradient boosting, test: f1
model                      llama  vicuna  gpt3.5   gpt4
generating_method                                      
DSN                        1.000   1.000     NaN    NaN
GCG                          NaN   0.857   0.571  0.000
JBC                          NaN   1.000     NaN    NaN
PAIR                         NaN   0.875   0.944  0.200
prompt_with_random_search  0.976   0.950   0.894  0.938

Gradient boosting, test: accuracy
model                      llama  vicuna  gpt3.5   gpt4
generating_method                                      
DSN                        1.000   1.000     NaN    NaN
GCG                        1.000   0.762   0.727  0.947
JBC                        1.000   1.000   1.000  1.000
PAIR                         NaN   0.789   0.895  0.333
prompt_with_random_search  0.952   0.905   0.808  0.889

Gradient boosting, test: precision
model                      llama  vicuna  gpt3.5   gpt4
generating_method                                      
DSN 

In [4]:
# %% Item 3: does mixing all four LLMs together help?
# Compares one model trained on all four LLMs with a separate model for each LLM.
# Both use the train and test rows from item 1, so each LLM's test rows are scored both ways.
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, LogisticRegression
 
llm_order = ["llama", "vicuna", "gpt3.5", "gpt4"]
 
# Same interaction columns as cv_v2 (they're already in the CSV)
interaction_cols = [f"{b}_x_{n}" for b in bench_cols for n in numeric_cols]
interaction_cols += [f"{b}_x_{m}" for b in bench_cols for m in method_cols]
 
# Features for each kind of model and training setup.
# Benchmarks and their interactions have one value per LLM, so they're left out when training on one LLM
linear_features = {"All four LLMs": numeric_cols + bench_cols + interaction_cols + method_cols[1:],   # same as cv_v2
                   "Own LLM only":  numeric_cols + method_cols[1:]}
tree_features = {"All four LLMs": feature_sets["All features"],
                 "Own LLM only":  feature_sets["No benchmarks"]}
 
item3_models = {
    "OLS":      (LinearRegression(), linear_features),
    "Logistic": (LogisticRegression(C=5.0, max_iter=1000), linear_features),
}
for model_name, model in models.items():
    item3_models[model_name] = (model, tree_features)
 
 
def scale_and_clip(fit_df, other_df, cols):
    """Fit the scaler on fit_df only, then scale and clip both."""
    scaler = StandardScaler()
    fit_df, other_df = fit_df.copy(), other_df.copy()
    fit_df[cols] = np.clip(scaler.fit_transform(fit_df[cols]), -2, 2)
    other_df[cols] = np.clip(scaler.transform(other_df[cols]), -2, 2)
    return fit_df, other_df
 
 
def get_scores(train_part, test_part, training):
    """Test scores from every ML model, one row per test prompt."""
    y_part = train_part["jailbroken"].astype(int)
    output = []
    for model_name, (model, feature_lists) in item3_models.items():
        features = feature_lists[training]
        cols = [c for c in features if not c.startswith("method_")]   # scale everything except the method columns
        train_rows, test_rows = scale_and_clip(train_part, test_part, cols)
        fitted = clone(model).fit(train_rows[features], y_part)
 
        # OLS has no predict_proba, so use its raw prediction, as in cv_v2
        if model_name == "OLS":
            score = fitted.predict(test_rows[features])
        else:
            score = fitted.predict_proba(test_rows[features])[:, 1]
 
        output.append(pd.DataFrame({"llm": test_part["model"],
                                    "ml_model": model_name,
                                    "training": training,
                                    "actual": test_part["jailbroken"].astype(int),
                                    "score": score}))
    return pd.concat(output)
 
 
# All four LLMs together
pooled = get_scores(train_df, test_df, "All four LLMs")
 
# One LLM at a time
single = []
for llm in llm_order:
    own_train = train_df[train_df["model"] == llm]
    own_test = test_df[test_df["model"] == llm]
    single.append(get_scores(own_train, own_test, "Own LLM only"))
 
# "all" scores every test row together, so the four separate models can be compared with one pooled model
all_scores = pd.concat([pooled] + single)
all_scores = pd.concat([all_scores, all_scores.assign(llm="all")])
 
mixing_rows = []
for (training, ml_model, llm), group in all_scores.groupby(["training", "ml_model", "llm"]):
    mixing_rows.append({"training": training, "ml_model": ml_model, "llm": llm,
                        **get_metrics(group["actual"], group["score"])})
mixing = pd.DataFrame(mixing_rows)
 
 
def show_mixing(metric):
    """Two lines per LLM (pooled and own-LLM training), one column per ML model."""
    table = mixing.pivot(index=["llm", "training"], columns="ml_model", values=metric)
    print(f"\ntest: {metric}")
    print(table.loc[llm_order + ["all"], list(item3_models)].round(3).to_string())
 
 
# Check: for the tree models, the "all / All four LLMs" rows should match the "All features" rows in item 1
for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    show_mixing(metric)


test: accuracy
ml_model                OLS  Logistic  Decision tree  Random forest  Gradient boosting  XGBoost
llm    training                                                                                
llama  All four LLMs  0.975     0.951          0.951          0.975              0.988    0.988
       Own LLM only   0.951     0.951          0.951          0.975              0.938    0.975
vicuna All four LLMs  0.847     0.857          0.837          0.847              0.888    0.857
       Own LLM only   0.847     0.857          0.847          0.847              0.867    0.857
gpt3.5 All four LLMs  0.779     0.802          0.826          0.826              0.849    0.791
       Own LLM only   0.791     0.791          0.779          0.779              0.779    0.802
gpt4   All four LLMs  0.825     0.841          0.857          0.810              0.825    0.825
       Own LLM only   0.889     0.857          0.825          0.810              0.873    0.873
all    All four LLMs  0.